In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import pipeline
from datasets import load_dataset
from huggingface_hub import list_repo_files
import numpy as np

In [ ]:
files = list_repo_files("HuggingFaceM4/WebSight", repo_type="dataset")
train_files = [f"https://huggingface.co/datasets/HuggingFaceM4/WebSight/resolve/main/{f}"
               for f in files if "data/train" in f and f.endswith(".parquet")]

In [ ]:
dataset = load_dataset("parquet", data_files={"train": train_files[:2]})

In [ ]:
dataset_xs = dataset["train"].shard(num_shards=100, index=0)

In [ ]:
sample_img = dataset_xs[1]["image"]
sample_img

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, PreTrainedModel
from transformers import Blip2VisionModel, Blip2QFormerModel
from transformer import Trainer, TrainingArguments

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [ ]:
class ImageTextCollator:
    def __init__(self, blip_processor, qwen_tokenizer):
        self.blip_processor = blip_processor
        self.qwen_tokenizer = qwen_tokenizer

    def __call__(self, batch):
        images = [x["image"] for x in batch]
        html = [x["text"] for x in batch]
        # ocr = [x["ocr"] for x in batch]

        blip_inputs = self.blip_processor(images=images, return_tensors="pt")
        qwen_html_inputs = self.qwen_tokenizer(html, return_tensors="pt", padding="longest")
        # qwen_ocr_inputs = self.qwen_tokenizer(ocr, return_tensors="pt", padding="longest")

        return {
            "pixel_values": blip_inputs["pixel_values"],
            "html_ids": qwen_html_inputs["input_ids"],
            # "ocr_ids": qwen_ocr_inputs["input_ids"]
        }

In [ ]:
vit_model_name = "google/vit-base-patch16-224"
vit_processor = ViTImageProcessor.from_pretrained(vit_model_name)
vit_model = ViTModel.from_pretrained(vit_model_name, device_map="auto")

In [ ]:
class

In [ ]:
model_name = "Qwen/Qwen3-4B-Instruct-2507"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    quantization_config=bnb_config,
    device_map="auto"
)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
class ImageTextModel(PreTrainedModel):
    def __init__(self, blip_model_name, qwen_model_name, quantization_config, lora_config, prompt="answer only in html"):
        super().__init__()

        self.blip_model = Blip2Model.from_pretrained(blip_model_name, device_map="auto", quantization_config=quantization_config)
        self.qwen_model = AutoModelForCausalLM.from_pretrained(
            qwen_model_name,
            quantization_config=quantization_config,
            device_map="auto"
        )
        self.qwen_model = prepare_model_for_kbit_training(self.qwen_model, use_gradient_checkpointing=True)
        self.qwen_model.gradient_checkpointing_enable()
        self.qwen_model = get_peft_model(self.qwen_model, lora_config)

        self.blip_tokenizer = Blip2Processor.from_pretrained(blip_model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
        self.proj = nn.Linear(self.blip_qformer_model.config.hidden_size, self.qwen_model.config.hidden_size)

        for p in self.blip_model.parameters():
            p.requires_grad = False

        prompt_ids = self.tokenizer(prompt, return_tensors="pt").input_ids
        self.prompt_embeds = self.qwen_model.model.embed_tokens(prompt_ids)


    def get_image_embeds(self, pixel_values):
        qformer_embeds = self.blip_qformer_model(pixel_values, return_dict=True).last_hidden_state
        image_embeds = self.proj(qformer_embeds)
        return image_embeds

    def forward(self, pixel_values, query_ids, html_ids=None, ocr_ids=None):
        batch_size = pixel_values.shape[0]
        image_embeds = get_image_embeds(pixel_values)

        # crafting the input
        prompt_expanded = self.prompt_embeds.expand(batch_size, -1, -1)
        html_embeds = self.qwen_model.model.embed_tokens(html_ids)

        if ocr_ids is not None:
            ocr_embeds = self.qwen_model.model.embed_tokens(ocr_ids)
            input_embeds = torch.cat([prompt_embeds, image_embeds, ocr_embeds], dim=1)
        else:
            input_embeds = torch.cat([prompt_embeds, image_embeds], dim=1)

        ignore_prefix = torch.full(
            (batch_size, input_embeds.shape[1]), -100, dtype=torch.long, device=html_ids.device
        )
        if html_ids is not None: #training
            input_embeds = torch.cat([input_embeds, html_ids], dim=1)
            labels = torch.cat([ignore_prefix, html_ids], dim=1)
            outputs = self.qwen_model(
                input_embeds=input_embeds,
                labels=labels
            )
            return outputs
        else: #generation
            outputs = self.qwen_model(inputs_embeds=input_embeds)
            return outputs

In [ ]:
vit_model_name = "google/vit-base-patch16-224"
qwen_model_name = "Qwen/Qwen3-4B-Instruct-2507"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_up_proj", "down_proj"]
)

model = ImageTextModel(vit_model_name, qwen_model_name, bnb_config, lora_config)
for p in model.parameters():
    p.requires_grad = False
for n, p in model.qwen_model.named_parameters():
    if "lora_" in n:
        p.requires_grad = True
for p in model.proj.parameters():
    p.requires_grad = True

print("Trainable params:",
      sum(p.numel() for p in base.parameters() if p.requires_grad) / 1e6, "M")

vit_processor = ViTImageProcessor.from_pretrained(vit_model_name)
collator = ImageTextCollator(vit_processor, model.tokenizer)


In [ ]:
tokenizer = processor.tokenizer

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "describe the image"},
        ],
    },
]
image = sample_img

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda")

# Generate
generate_ids = model.generate(inputs.input_ids, max_new_tokens=1500)
tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

KeyboardInterrupt: 

In [ ]:
content

' for a website.\n\n<!DOCTYPE html>\n<html lang="en">\n<head>\n    <meta charset="UTF-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <title>Intro Page</title>\n</head>\n<body>\n    <h1>Welcome to Our Website</h1>\n    <p>This is a simple intro page for a website.</p>\n</body>\n</html>'